# 11 · Triage: which parts of a video are worth preparing

A quick pass over a whole video that scores how usable each moment is for reconstructing the
main person, so notebook 10 only runs on the good stretches. It is built to be cheap:

1. **Sample cheaply.** By default only the video's keyframes are decoded (every 2-4 s in typical
   files; decoding them skips the frames between, about 150x real time even for 4K). Or every
   `EVERY_N`-th frame, which decodes everything.
2. **Cheapest model first.** YOLO-pose finds people and 17 joints in a few milliseconds. Samples
   with nobody in them, or someone too small or too hidden, stop there.
3. **Expensive check last.** Only the samples that pass get Sapiens2's body parts, at half
   resolution, to see whether another body touches the main person: the other person is often
   barely in frame (no head, just a leg) and no person detector finds them, but their skin is
   still labelled as a body part.

The main person is the largest one in each sample. Each sample gets a score from 0 to 1 and the
reasons that lowered it; neighbouring samples that look alike (colour histograms) are grouped into
shots, and the best shots are printed as ready settings for notebook 10.

In [ ]:
import time
from contextlib import contextmanager
from pathlib import Path

import av
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from transformers import AutoImageProcessor, Sapiens2ForSemanticSegmentation
from ultralytics import YOLO

from avatar_splats.preprocess.triage import rank_shots

device = "cuda"

## Settings

The thresholds below are starting points; look at what the contact sheets reject and adjust.

In [ ]:
VIDEO = Path("../examples/live-performance/6.mp4")  # relative to notebooks/
SAMPLING = "keyframes"  # "keyframes" (fast) or "every_n" (decodes every frame)
EVERY_N = 25  # with "every_n": every this many frames
MAX_KEYFRAME_GAP = 6.0  # seconds: sparser keyframes fall back to "every_n"
MAX_SAMPLES = 5000  # stop after this many samples
ANALYSIS_SIDE = 640  # samples are scaled so their longer side is this many pixels

POSE_MODEL = "../assets/yolo/yolo11m-pose.pt"  # downloads on first use; -n/-s/-l/-x sizes too
PARTS_MODEL = "facebook/sapiens2-seg-1b"
PARTS_SIZE = (512, 384)  # Sapiens2's input, half its native 1024 x 768: about 4x faster
BATCH = 16

MIN_PERSON = 0.5  # YOLO's score to count a person
MIN_HEIGHT = 0.25  # the main person's box height, in frame heights, to be worth anything
GOOD_HEIGHT = 0.6  # and to count fully
MIN_JOINTS = 6  # joints seen (confidence >= JOINT_SEEN) to go on to the contact check
JOINT_SEEN = 0.5
EDGE = 0.01  # a box this near the frame's edge (share of its side) counts as cut off there
CONTEXT = 1.6  # the contact check looks at a crop this many times the person's box
MAX_CONTACT = 0.3  # other body pixels per pixel of hers that count as full contact
SHOT_CUT = 0.45  # colour histogram distance (Bhattacharyya) that starts a new shot
GOOD, OK = 0.6, 0.3  # score thresholds of the labels

timings = {}


@contextmanager
def timed(step):
    torch.cuda.synchronize()
    started = time.perf_counter()
    yield
    torch.cuda.synchronize()
    timings[step] = timings.get(step, 0.0) + time.perf_counter() - started

## Samples

In [ ]:
def decode(sampling):
    # (frames at ANALYSIS_SIDE, their times in s, fps, duration in s or None, original size)
    frames, times = [], []
    with av.open(str(VIDEO)) as container:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        if sampling == "keyframes":
            stream.codec_context.skip_frame = "NONKEY"
        scale = ANALYSIS_SIDE / max(stream.width, stream.height)
        size = (round(stream.width * scale) // 2 * 2, round(stream.height * scale) // 2 * 2)
        for index, frame in enumerate(container.decode(stream)):
            if sampling == "every_n" and index % EVERY_N:
                continue
            frames.append(
                frame.reformat(width=size[0], height=size[1], format="rgb24").to_ndarray()
            )
            times.append(float(frame.pts * stream.time_base))
            if len(frames) >= MAX_SAMPLES:
                break
        duration = float(stream.duration * stream.time_base) if stream.duration else None
        return (
            frames,
            np.array(times),
            float(stream.average_rate),
            duration,
            (stream.width, stream.height),
        )


with timed("decode"):
    sampling = SAMPLING
    samples, sample_times, fps, duration, original = decode(sampling)
    gaps = np.diff(sample_times)
    if sampling == "keyframes" and (len(samples) < 10 or np.median(gaps) > MAX_KEYFRAME_GAP):
        print(f"only {len(samples)} keyframes, too sparse: every {EVERY_N}th frame instead")
        sampling = "every_n"
        samples, sample_times, fps, duration, original = decode(sampling)
        gaps = np.diff(sample_times)
COUNT, (HEIGHT, WIDTH) = len(samples), samples[0].shape[:2]
length = duration or sample_times[-1]
print(f"{VIDEO.name}: {original[0]}x{original[1]}, {fps:g} fps, {length:.0f} s")
spacing = f"every {np.median(gaps):.1f} s (most {gaps.max():.1f} s)"
print(f"{COUNT} samples ({sampling}) at {WIDTH}x{HEIGHT}, {spacing}")

## Shots

Each sample's colours (a small HSV histogram) are compared with the previous sample's; a
distance above `SHOT_CUT` starts a new shot. With keyframe sampling a cut is only known to lie
between two samples, a few seconds apart.

In [ ]:
with timed("shots"):
    histograms = []
    for sample in samples:
        hsv = cv2.cvtColor(
            cv2.resize(sample, (64, 36), interpolation=cv2.INTER_AREA), cv2.COLOR_RGB2HSV
        )
        histogram = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 4], [0, 180, 0, 256, 0, 256])
        histograms.append(cv2.normalize(histogram, None).flatten())
    change = np.array(
        [0.0]
        + [
            cv2.compareHist(a, b, cv2.HISTCMP_BHATTACHARYYA)
            for a, b in zip(histograms, histograms[1:], strict=False)
        ]
    )
shot_of = np.cumsum(change > SHOT_CUT)
SHOTS = shot_of[-1] + 1
print(f"{SHOTS} shots, {COUNT / SHOTS:.1f} samples per shot on average")

## Stage 1: people and joints (YOLO-pose), every sample

The main person is the largest box. Per sample: whether anyone is there, how tall the main
person is, how many of the 17 joints show, whether their box touches the frame's edge, whether
other people overlap them, and how sharp they are (variance of the Laplacian over their box,
compared with the video's median, so the score adapts to the footage).

In [ ]:
pose_model = YOLO(POSE_MODEL)
keypoints = np.full((COUNT, 17, 3), np.nan)
boxes = np.full((COUNT, 5), np.nan)  # x0, y0, x1, y1, score
people = np.zeros(COUNT, int)
overlap = np.zeros(COUNT)  # the largest share of the main box another person's box covers
with timed("people and joints (YOLO-pose)"):
    for start in range(0, COUNT, BATCH):
        results = pose_model.predict(
            samples[start : start + BATCH],
            imgsz=ANALYSIS_SIDE,
            quantize=16,
            verbose=False,
            device=0,
        )
        for i, result in enumerate(results, start=start):
            found = result.boxes.data.cpu().numpy()  # x0, y0, x1, y1, score, class
            joints = result.keypoints.data.cpu().numpy()  # (people, 17, 3)
            keep = found[:, 4] >= MIN_PERSON
            found, joints = found[keep], joints[keep]
            people[i] = len(found)
            if not len(found):
                continue
            areas = (found[:, 2] - found[:, 0]) * (found[:, 3] - found[:, 1])
            main = int(areas.argmax())
            boxes[i], keypoints[i] = found[main, :5], joints[main]
            x0, y0, x1, y1 = found[main, :4]
            for other in np.delete(found, main, axis=0):
                w = max(0, min(x1, other[2]) - max(x0, other[0]))
                h = max(0, min(y1, other[3]) - max(y0, other[1]))
                overlap[i] = max(overlap[i], w * h / areas[main])

with timed("sharpness"):
    sharpness = np.full(COUNT, np.nan)
    for i in np.flatnonzero(people > 0):
        x0, y0, x1, y1 = boxes[i, :4].astype(int)
        grey = cv2.cvtColor(samples[i][max(y0, 0) : y1, max(x0, 0) : x1], cv2.COLOR_RGB2GRAY)
        if grey.size:
            grey = cv2.resize(grey, (max(1, round(256 * grey.shape[1] / grey.shape[0])), 256))
            sharpness[i] = cv2.Laplacian(grey, cv2.CV_32F).var()

height = (boxes[:, 3] - boxes[:, 1]) / HEIGHT
joints_seen = (np.nan_to_num(keypoints[..., 2]) >= JOINT_SEEN).sum(-1)
edges = np.stack(
    [
        boxes[:, 0] <= EDGE * WIDTH,
        boxes[:, 1] <= EDGE * HEIGHT,
        boxes[:, 2] >= (1 - EDGE) * WIDTH,
        boxes[:, 3] >= (1 - EDGE) * HEIGHT,
    ],
    1,
)
cut_off = edges.any(1) & (joints_seen < 17)
survivors = (people > 0) & (np.nan_to_num(height) >= MIN_HEIGHT) & (joints_seen >= MIN_JOINTS)
print(f"someone in {(people > 0).sum()} of {COUNT} samples")
print(f"{survivors.sum()} go on to the contact check")

## Stage 2: contact (Sapiens2, half resolution), survivors only

Sapiens2 labels every body pixel in a crop `CONTEXT` times the main person's box, without
having to find anyone first. Body pixels outside the main person's box (slightly widened) belong
to someone else, however little of them shows; their number per body pixel inside the box is
the contact. It misses another body lying entirely inside her box, which overlapping boxes
(stage 1) partly catch.

In [ ]:
parts_processor = AutoImageProcessor.from_pretrained(PARTS_MODEL)
parts_model = (
    Sapiens2ForSemanticSegmentation.from_pretrained(PARTS_MODEL, torch_dtype=torch.bfloat16)
    .to(device)
    .eval()
)
CLOTHING = (
    1,
    2,
    9,
    10,
    13,
    18,
    19,
    23,
)  # apparel, eyeglass, shoes, socks, lower and upper clothing
contact = np.full(COUNT, np.nan)
parts_crops = {}


def context_crop(box):
    # x0, y0, x1, y1 of a 3:4 crop CONTEXT times the box, inside the frame where possible
    x0, y0, x1, y1 = box[:4]
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    h = CONTEXT * max(y1 - y0, (x1 - x0) * 4 / 3)
    w = h * 3 / 4
    return [round(cx - w / 2), round(cy - h / 2), round(cx + w / 2), round(cy + h / 2)]


def cut(image, crop):
    x0, y0, x1, y1 = crop
    out = np.full((y1 - y0, x1 - x0, 3), (124, 116, 104), np.uint8)
    fx0, fy0, fx1, fy1 = max(x0, 0), max(y0, 0), min(x1, WIDTH), min(y1, HEIGHT)
    out[fy0 - y0 : fy1 - y0, fx0 - x0 : fx1 - x0] = image[fy0:fy1, fx0:fx1]
    return out


with timed("contact (Sapiens2)"), torch.inference_mode():
    todo = np.flatnonzero(survivors)
    for start in range(0, len(todo), BATCH):
        batch = todo[start : start + BATCH]
        crops = [context_crop(boxes[i]) for i in batch]
        inputs = parts_processor(
            images=[cut(samples[i], c) for i, c in zip(batch, crops, strict=True)],
            size={"height": PARTS_SIZE[0], "width": PARTS_SIZE[1]},
            return_tensors="pt",
        )
        labels = parts_model(**inputs.to(device, torch.bfloat16)).logits.argmax(1).cpu().numpy()
        for i, crop, label in zip(batch, crops, labels, strict=True):
            body = (label > 0) & ~np.isin(label, CLOTHING)
            x0, y0, x1, y1 = crop
            sx, sy = PARTS_SIZE[1] / (x1 - x0), PARTS_SIZE[0] / (y1 - y0)
            bx0, by0, bx1, by1 = boxes[i, :4]
            margin = 0.05 * max(bx1 - bx0, by1 - by0)
            inside = np.zeros_like(body)
            inside[
                max(0, int((by0 - margin - y0) * sy)) : int((by1 + margin - y0) * sy),
                max(0, int((bx0 - margin - x0) * sx)) : int((bx1 + margin - x0) * sx),
            ] = True
            contact[i] = (body & ~inside).sum() / max((body & inside).sum(), 1)
            parts_crops[i] = (crop, label)
print(
    f"contact measured on {np.isfinite(contact).sum()} samples, median {np.nanmedian(contact):.2f}"
)

## Scores

Each sample's score multiplies five shares in [0, 1]; the lowest ones are its reasons:

- **size**: the main person's height between `MIN_HEIGHT` and `GOOD_HEIGHT`,
- **joints**: joints seen / 17,
- **whole**: 1, or 0.5 if their box touches the frame's edge with joints missing,
- **sharpness**: their sharpness / the video's median, capped at 1,
- **alone**: 1 minus the contact (or the overlap with another person's box, whichever is
  larger) / `MAX_CONTACT`, at least 0.

Samples without anyone, or stopped at stage 1, score 0.

In [ ]:
FACTORS = ("size", "joints", "whole", "sharpness", "alone")
factors = np.zeros((COUNT, len(FACTORS)))
factors[:, 0] = np.clip((np.nan_to_num(height) - MIN_HEIGHT) / (GOOD_HEIGHT - MIN_HEIGHT), 0, 1)
factors[:, 1] = joints_seen / 17
factors[:, 2] = np.where(cut_off, 0.5, 1.0)
factors[:, 3] = np.clip(np.nan_to_num(sharpness / np.nanmedian(sharpness)), 0, 1)
factors[:, 4] = np.clip(1 - np.maximum(np.nan_to_num(contact), overlap) / MAX_CONTACT, 0, 1)
score = np.where(survivors, factors.prod(1), 0.0)
label = np.where(score >= GOOD, "good", np.where(score >= OK, "ok", "bad"))


def reasons(i):
    if people[i] == 0:
        return "nobody"
    if not survivors[i]:
        return "too small" if np.nan_to_num(height[i]) < MIN_HEIGHT else "too few joints"
    weakest = np.argsort(factors[i])[:2]
    return (
        ", ".join(f"{FACTORS[k]} {factors[i, k]:.2f}" for k in weakest if factors[i, k] < 0.9)
        or "-"
    )


for name in ("good", "ok", "bad"):
    print(f"{name:>5}: {(label == name).sum():5d} samples ({(label == name).mean():.0%})")

## Over time

In [ ]:
minutes = sample_times / 60
fig, axes = plt.subplots(len(FACTORS) + 1, 1, figsize=(16, 2 * (len(FACTORS) + 1)), sharex=True)
colours = {"good": "tab:green", "ok": "tab:orange", "bad": "tab:red"}
for name, colour in colours.items():
    chosen = label == name
    axes[0].scatter(minutes[chosen], score[chosen], s=6, color=colour, label=name)
for cut_at in minutes[1:][np.diff(shot_of) > 0]:
    axes[0].axvline(cut_at, color="grey", lw=0.3)
axes[0].set_ylabel("score")
axes[0].legend(loc="upper right")
for ax, k in zip(axes[1:], range(len(FACTORS)), strict=True):
    ax.plot(minutes, factors[:, k], lw=0.6)
    ax.set_ylabel(FACTORS[k])
    ax.set_ylim(-0.05, 1.05)
axes[-1].set_xlabel("minutes")
plt.tight_layout()
plt.show()

## Contact sheets: the best and the worst samples

The main person's box (green), their joints (red to green by confidence), other people (orange)
and, where the contact check ran, other people's body pixels (magenta).

In [ ]:
BONES = ((0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12),
         (11, 12), (11, 13), (13, 15), (12, 14), (14, 16))  # fmt: skip


def annotated(i):
    image = samples[i].copy()
    if i in parts_crops:
        (x0, y0, x1, y1), parts = parts_crops[i]
        body = ((parts > 0) & ~np.isin(parts, CLOTHING)).astype(np.uint8)
        body = cv2.resize(body, (x1 - x0, y1 - y0), interpolation=cv2.INTER_NEAREST).astype(bool)
        full = np.zeros((HEIGHT, WIDTH), bool)
        fx0, fy0, fx1, fy1 = max(x0, 0), max(y0, 0), min(x1, WIDTH), min(y1, HEIGHT)
        full[fy0:fy1, fx0:fx1] = body[fy0 - y0 : fy1 - y0, fx0 - x0 : fx1 - x0]
        bx0, by0, bx1, by1 = boxes[i, :4].astype(int)
        full[max(by0, 0) : by1, max(bx0, 0) : bx1] = False
        image[full] = (0.5 * image[full] + 0.5 * np.array([255, 0, 255])).astype(np.uint8)
    if people[i]:
        x0, y0, x1, y1 = boxes[i, :4].astype(int)
        cv2.rectangle(image, (x0, y0), (x1, y1), (0, 220, 0), 2)
        points = keypoints[i]
        for a, b in BONES:
            if min(points[a, 2], points[b, 2]) >= 0.3:
                c = min(points[a, 2], points[b, 2])
                cv2.line(
                    image,
                    points[a, :2].astype(int),
                    points[b, :2].astype(int),
                    (int(255 * (1 - c)), int(255 * c), 0),
                    2,
                )
    return image


def sheet(indices, title):
    columns = 4
    rows = max(1, int(np.ceil(len(indices) / columns)))
    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(18, 18 / columns * HEIGHT / WIDTH * rows + 0.6 * rows),
        squeeze=False,
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, indices, strict=False):
        ax.imshow(annotated(i))
        ax.set_title(f"{sample_times[i] / 60:.1f} min · {score[i]:.2f} · {reasons(i)}", fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


ranked = np.argsort(-score)
sheet(ranked[:12], "best")
sheet([i for i in ranked[::-1] if people[i] > 0][:12], "worst with someone in them")

## Shots, best first, as settings for notebook 10

A shot's score is the median of its samples'. Notebook 10 reads `triage.npz` (saved below) and
prepares the shot whose rank its `SHOT` names, 0 the best; with `TRIAGE = None` there, `START` is
the shot's first frame.

In [ ]:
shots = rank_shots(sample_times, shot_of, score, length)

print(f"{'SHOT':>4}  {'score':>6}  {'from':>8}  {'to':>8}  {'length':>7}  samples  START")
for rank, shot in enumerate(shots[:20]):
    times = f"{shot.begin / 60:6.1f} m  {shot.end / 60:6.1f} m  {shot.end - shot.begin:6.0f} s"
    start = round(shot.begin * fps)
    print(f"{rank:4d}  {shot.score:6.2f}  {times}  {len(shot.samples):7d}  {start}")

## Save, and where the time went

In [ ]:
out_dir = Path("../outputs/triage") / VIDEO.stem
out_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    out_dir / "triage.npz",
    times=sample_times, score=score, factors=factors, factor_names=np.array(FACTORS), label=label,
    shot=shot_of, people=people, boxes=boxes, keypoints=keypoints, contact=contact, overlap=overlap,
    sharpness=sharpness,
)  # fmt: skip
print("written to", out_dir / "triage.npz")

total = sum(timings.values())
print(f"\n{'step':<30}{'seconds':>9}{'ms/sample':>11}{'share':>8}")
for step, seconds in sorted(timings.items(), key=lambda item: -item[1]):
    print(f"{step:<30}{seconds:>9.1f}{1000 * seconds / COUNT:>11.1f}{seconds / total:>8.0%}")
print(f"{'total':<30}{total:>9.1f}{1000 * total / COUNT:>11.1f}")
speed = length / total
print(f"\n{length / 60:.0f} min of video triaged in {total / 60:.1f} min ({speed:.0f}x real time)")